# 01 ? Fixed-split training

Train 3DR-Net or the existing standard 3D U-Net on an explicit animal-level split.
The shared experiment core owns loading, preprocessing, compilation, checkpoint selection,
and optional held-out full-volume evaluation. Run with the **rnet** environment.

Start by reviewing the configuration and split counts. Set `RUN_TRAINING = True` when ready.
This notebook starts a fresh experiment; it does not resume interrupted training.
The historical notebooks remain references. No architectural defaults or dropout rates are changed.

## 1. Workspace and runtime
Configure GPU memory and precision before constructing generators or models.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime

## 2. Experiment configuration

The default task uses D1, with left/right lateral ventricles merged into one class,
as in the historical training notebook. D1 also contains fully annotated scans;
`unknown="background"` explicitly projects their additional anatomy labels into task background. Normalized metadata supplies annotation availability.
Reorientation uses the frozen reference below, before all other preprocessing.
Existing N4 images are loaded; bias correction is not repeated.

All stages resample onto an orthogonal grid before augmentation, removing source affine shear
while retaining the physical field of view. Images use linear interpolation and masks use nearest neighbour. Validation uses a fixed
centre crop and no augmentation. Full-volume testing does not crop the input.
`random_crop=True` opts into uniform training patch sampling; the default uses fixed centre crops.

For partial labels, select `dice_ce_gradient_fix` and explicitly configure the complete source
mapping and class names. The core trains ROI logits and exports softmax inference probabilities.
Legacy sources require an explicit modality and `available_labels` in `DatasetSpec`.

In [ ]:
from src.experiments.config import (
    DataConfig, DatasetSpec, ExperimentConfig, InferenceConfig,
    LabelConfig, LossConfig, ModelConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
REFERENCE: Path = DATA_ROOT / "D1" / "SUBJ_0001" / "N4.nii.gz"
RUN_TRAINING: bool = False

training = TrainingConfig(
    model_name="d1_3drnet", epochs=300, batch_size=8, seed=TRAINING_SEED,
    output_dir=str(REPO_ROOT / "results"), fit_kwargs={"verbose": 1},
)
data_settings = DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D1"),))
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(80, 80, 80), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=False,
)
model_settings = ModelConfig(architecture="3drnet", filters=(16, 32, 64, 128, 256))
loss_settings = LossConfig(regions="dice_ce", gamma=0.8, brain_alpha=0.6, batch_wise=True)
inference_settings = InferenceConfig(enabled=True, stride=40, save_predictions=True)

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    model=model_settings, labels=LabelConfig(unknown="background"), loss=loss_settings,
    inference=inference_settings, learning_rate=0.001, split_seed=42,
    early_stopping_patience=20,
)
config.validate()

## 3. Load datasets and select animals

Provide an existing `SPLIT_FILE` for matched or predefined experiments. Otherwise this cell
creates a reproducible, unstratified split using fractions of the complete animal cohort.
Every selected animal and all its timepoints belong to exactly one role.
Review cohort composition before training; D2 needs a separately designed small-cohort protocol.
The exact split and original-ID manifest are saved with the run.

In [ ]:
from IPython.display import display
from src.experiments.data import SubjectSplit, make_split, prepare_data

SPLIT_FILE: Path | None = None
prepared = prepare_data(config)

if SPLIT_FILE is None:
    split = make_split(
        prepared.subjects, seed=config.split_seed,
        validation_fraction=0.2, test_fraction=0.1,
    )
else:
    split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)

display(prepared.split_summary(split))
display(prepared.subjects.manifest(list(split.train)).head())

## 4. Check one validation batch

This optional preparation check loads one batch without training a model. Validation retains
its final incomplete batch and never repeats scans. Full-volume manuscript metrics are distinct
from these centre-crop training diagnostics.

In [ ]:
validation_generator = prepared.make_generator(split.validation, training=False, batch_size=1)
images, targets = validation_generator[0]

print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Validation batches:", len(validation_generator))

del images, targets, validation_generator

## 5. Train and evaluate

The runner restores the best validation checkpoint before optional held-out test inference.
Test results are not used for checkpoint selection. Each execution creates a separate run folder.
Prediction files use the original image grid with **output class IDs**, including the merged
ventricular class. Physical metrics use the complete aligned resampled grid.
No anatomical postprocessing is enabled in this first workflow.

In [ ]:
from src.experiments.training import run_training

result = None
if RUN_TRAINING:
    result = run_training(prepared, split)
    print("Run saved to:", result.run_dir)
else:
    print("Configuration checked. Set RUN_TRAINING = True and rerun this cell to train.")

## 6. Inspect results
Plotly uses the native VS Code renderer; no widgets or embedded HTML are needed.

In [ ]:
import pandas as pd
import plotly.io as pio
from src.experiments.training import history_figure

pio.renderers.default = "vscode"

if result is not None:
    figure = history_figure(result)
    figure.show()
    display(pd.DataFrame([result.summaries]))

    if result.evaluations.get("regions"):
        metrics = pd.concat(result.evaluations["regions"], ignore_index=True)
        dice = metrics.loc[metrics["Metric"] == "Dice"]
        display(dice[["Subject", "Scan", "Region", "Value", "Status"]])

## Saved artifacts and next steps

The run contains `experiment.json`, `metadata.json`, `subjects.csv`, and `splits.json`.
`fold_0/` contains the best weights, an inference model, training history, and timings.
When enabled, test evaluation includes regional metric tables, per-scan inference timings,
and original-grid ROI/brain predictions. Status columns distinguish unavailable annotations
from annotated empty structures; inspect valid counts before summarizing results.

For another architecture, change `ModelConfig.architecture` and the experiment name while
reusing the same saved split. Keep other settings fixed for a controlled comparison.
This notebook does not perform ablations, statistical comparisons, fine-tuning, or sensitivity analysis.
See `docs/experiments.md` for the core API and current limitations.